# Sales Performance Dashboard — Data Cleaning & Feature Engineering

**Dataset:** Superstore Sales Dataset (Kaggle)
**Purpose:** Clean raw transactional sales data and engineer the features used
in EDA, SQL analysis, and the Power BI dashboard.

**Pipeline steps covered in this notebook:**
1. Load the raw dataset
2. Diagnose and fix a structural data-quality issue (multi-sheet export)
3. Handle missing values and duplicates
4. Fix data types
5. Apply business-rule validation filters
6. Flag statistical outliers (IQR method)
7. Engineer 12 new features (profit margin, calendar fields, tiers, etc.)
8. Validate the result and export the cleaned dataset

## 1. Imports and configuration

In [1]:
from __future__ import annotations

import logging
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

RAW_PATH = Path("../data/raw/Superstore.csv")
CLEAN_PATH = Path("../data/cleaned/Superstore_Cleaned.csv")

SALES_BUCKET_EDGES = [0, 50, 200, 500, np.inf]
SALES_BUCKET_LABELS = ["Low (<$50)", "Medium ($50-$200)", "High ($200-$500)", "Premium ($500+)"]

DISCOUNT_TIER_EDGES = [-0.001, 0, 0.2, 0.4, 1.0]
DISCOUNT_TIER_LABELS = ["No Discount", "Low (0-20%)", "Medium (20-40%)", "High (40%+)"]

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)-8s | %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("data_cleaning")

## 2. Load the raw dataset

In [2]:
def load_raw_data(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Raw data file not found at {path.resolve()}")
    df = pd.read_csv(path, encoding="latin1")
    log.info("Loaded raw data: %s rows, %s columns", *df.shape)
    return df

## 3. Diagnose the structural data-quality issue

This raw export flattens **three separate Excel sheets** (Orders, People,
Returns) into one CSV. Rows belonging to the People/Returns sheets show up
as near-total nulls across every Orders column — they aren't missing data,
they simply don't belong in this table. `Row ID` is a clean sequential
integer that only the real Orders sheet has, so it's used as the structural
filter, rather than trying to impute values into rows that were never
Orders rows in the first place.

In [3]:
def isolate_orders_table(df: pd.DataFrame) -> pd.DataFrame:
    before = len(df)
    is_orders_row = pd.to_numeric(df["Row ID"], errors="coerce").notnull()
    df = df.loc[is_orders_row].copy()
    df["Row ID"] = df["Row ID"].astype(int)
    log.info(
        "Isolated Orders rows: kept %s of %s (%s non-Orders rows dropped)",
        len(df), before, before - len(df),
    )
    return df

## 4. Impute remaining missing values

Any partial nulls left (e.g. a missing Postal Code on an otherwise valid
Orders row) are imputed conservatively using the modal postal code for that
City/State pair — safer than a global median, which could assign a code
from an unrelated part of the country.

In [4]:
def impute_missing_postal_codes(df: pd.DataFrame) -> pd.DataFrame:
    missing_before = df["Postal Code"].isnull().sum()
    if missing_before:
        df["Postal Code"] = df.groupby(["State", "City"])["Postal Code"].transform(
            lambda x: x.fillna(x.mode().iloc[0] if not x.mode().empty else np.nan)
        )
        log.info("Imputed %s missing postal codes via City/State mode", missing_before)
    return df

## 5. Remove duplicate records

Full-row duplicates represent the same order line exported twice — a known
artifact of this dataset, not a genuine repeat purchase (a real repeat
purchase has a distinct Row ID).

In [5]:
def remove_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    before = len(df)
    df = df.drop_duplicates()
    log.info("Removed %s full-row duplicates", before - len(df))
    return df

## 6. Fix data types

Rows with unparseable dates are dropped and logged rather than silently
coerced to `NaT`, since every downstream time feature depends on a valid
Order Date.

In [6]:
def fix_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    df["Order Date"] = pd.to_datetime(df["Order Date"], format="%m/%d/%Y", errors="coerce")
    df["Ship Date"] = pd.to_datetime(df["Ship Date"], format="%m/%d/%Y", errors="coerce")

    before = len(df)
    df = df.dropna(subset=["Order Date", "Ship Date"])
    if before - len(df):
        log.warning("Dropped %s rows with unparseable Order/Ship Date", before - len(df))

    df["Postal Code"] = df["Postal Code"].astype("Int64")
    df["Quantity"] = df["Quantity"].astype(int)
    df["Sales"] = df["Sales"].astype(float).round(2)
    df["Profit"] = df["Profit"].astype(float).round(2)
    df["Discount"] = df["Discount"].astype(float).round(2)
    return df

## 7. Apply business-rule validation filters

- Sales/Quantity must be positive (a return/adjustment row is not a sale)
- Discount must be a proportion in [0, 1]
- Ship Date can never precede Order Date

In [7]:
def apply_business_rule_filters(df: pd.DataFrame) -> pd.DataFrame:
    before = len(df)
    df = df[(df["Sales"] > 0) & (df["Quantity"] > 0)]
    df = df[(df["Discount"] >= 0) & (df["Discount"] <= 1)]
    df = df[df["Ship Date"] >= df["Order Date"]]
    log.info("Business-rule filtering: kept %s of %s rows", len(df), before)
    return df

## 8. Flag statistical outliers (IQR method)

Outliers are **flagged, not deleted** — a \$10,000 order is a real,
valuable transaction, and deleting it would understate revenue. Flagging
lets EDA/BI tools filter outliers in or out on demand.

In [8]:
def flag_outliers_iqr(series: pd.Series) -> pd.Series:
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return (series < lower) | (series > upper)


def add_outlier_flags(df: pd.DataFrame) -> pd.DataFrame:
    df["Is_Sales_Outlier"] = flag_outliers_iqr(df["Sales"])
    df["Is_Profit_Outlier"] = flag_outliers_iqr(df["Profit"])
    log.info(
        "Flagged outliers — Sales: %s, Profit: %s",
        df["Is_Sales_Outlier"].sum(), df["Is_Profit_Outlier"].sum(),
    )
    return df

## 9. Feature engineering

All operations are vectorized (`pd.cut`, `.dt` accessors, `groupby.transform`)
— no per-row Python loops. 12 new columns are added here, including the
single most important derived metric for this project: **Profit Margin**.

In [9]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df["Profit_Margin"] = (df["Profit"] / df["Sales"]).round(4)

    df["Order_Year"] = df["Order Date"].dt.year
    df["Order_Month"] = df["Order Date"].dt.month
    df["Order_Month_Name"] = df["Order Date"].dt.strftime("%b")
    df["Order_Quarter"] = df["Order Date"].dt.quarter
    df["Order_Weekday"] = df["Order Date"].dt.day_name()

    df["Shipping_Days"] = (df["Ship Date"] - df["Order Date"]).dt.days

    df["Sales_Bucket"] = pd.cut(df["Sales"], bins=SALES_BUCKET_EDGES, labels=SALES_BUCKET_LABELS)
    df["Discount_Tier"] = pd.cut(df["Discount"], bins=DISCOUNT_TIER_EDGES, labels=DISCOUNT_TIER_LABELS)

    df["Is_Profitable"] = df["Profit"] > 0

    orders_per_customer = df.groupby("Customer ID")["Order ID"].transform("nunique")
    df["Customer_Type"] = np.where(orders_per_customer > 1, "Repeat", "One-Time")

    log.info("Feature engineering complete: %s columns total", df.shape[1])
    return df

## 10. Validate

Guard the invariants the rest of the project depends on — cheap, fast
checks that surface a silent upstream bug immediately instead of it
quietly propagating into every downstream chart, query, and dashboard.

In [10]:
def validate(df: pd.DataFrame) -> None:
    assert df["Row ID"].is_unique, "Duplicate Row IDs survived cleaning"
    assert df["Order Date"].notnull().all(), "Null Order Date survived cleaning"
    assert (df["Sales"] > 0).all(), "Non-positive Sales survived cleaning"
    log.info("Validation passed: all invariants hold on %s rows", len(df))

## 11. Run the full pipeline

In [11]:
def run_pipeline(raw_path: Path = RAW_PATH, clean_path: Path = CLEAN_PATH) -> pd.DataFrame:
    df = load_raw_data(raw_path)
    df = isolate_orders_table(df)
    df = impute_missing_postal_codes(df)
    df = remove_duplicates(df)
    df = fix_dtypes(df)
    df = apply_business_rule_filters(df)
    df = add_outlier_flags(df)
    df = engineer_features(df)
    validate(df)

    clean_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(clean_path, index=False)
    log.info("Cleaned dataset exported to %s (%s rows, %s columns)", clean_path, *df.shape)
    log.info("Date range: %s to %s", df["Order Date"].min().date(), df["Order Date"].max().date())
    return df

In [12]:
cleaned_df = run_pipeline()
cleaned_df.head()

08:47:02 | INFO     | Loaded raw data: 10800 rows, 21 columns


08:47:02 | INFO     | Isolated Orders rows: kept 9994 of 10800 (806 non-Orders rows dropped)


08:47:02 | INFO     | Imputed 11 missing postal codes via City/State mode


08:47:02 | INFO     | Removed 0 full-row duplicates


08:47:02 | INFO     | Business-rule filtering: kept 9994 of 9994 rows


...

08:47:02 | INFO     | Feature engineering complete: 34 columns total


08:47:02 | INFO     | Validation passed: all invariants hold on 9994 rows


08:47:02 | INFO     | Cleaned dataset exported to ../data/cleaned/Superstore_Cleaned.csv (9994 rows, 34 columns)


08:47:02 | INFO     | Date range: 2015-01-03 to 2018-12-30


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Is_Sales_Outlier,Is_Profit_Outlier,Profit_Margin,Order_Year,Order_Month,Order_Month_Name,Order_Quarter,Order_Weekday,Shipping_Days,Sales_Bucket,Discount_Tier,Is_Profitable,Customer_Type
0,1,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.00,41.91,False,False,0.1600,2017,11,Nov,4,Wednesday,3,High ($200-$500),No Discount,True,Repeat
1,2,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.00,219.58,True,True,0.3000,2017,11,Nov,4,Wednesday,3,Premium ($500+),No Discount,True,Repeat
2,3,CA-2017-138688,2017-06-12,2017-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0.00,6.87,False,False,0.4699,2017,6,Jun,2,Monday,4,Low (<$50),No Discount,True,Repeat
3,4,US-2016-108966,2016-10-11,2016-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.58,5,0.45,-383.03,True,True,-0.4000,2016,10,Oct,4,Tuesday,7,Premium ($500+),High (40%+),False,Repeat
4,5,US-2016-108966,2016-10-11,2016-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.37,2,0.20,2.52,False,False,0.1127,2016,10,Oct,4,Tuesday,7,Low (<$50),Low (0-20%),True,Repeat


## Result

Output: `data/cleaned/Superstore_Cleaned.csv` — 9,994 rows × 34 columns,
ready for the EDA notebook (`02_eda.ipynb`) and the SQL star schema build.